# Ontology builder evaluation: walkthrough and checks

Exercises the pieces of the evaluation step by step (run with the backend venv, e.g. a kernel on
`backend/.venv`, from this folder):

1. reference ontologies (`references/*.ttl`) and the scoring of a hand-made ontology,
2. the users' saved sessions (`compare-sessions.py` without KG rebuild),
3. the gold annotations of GutBrainIE and LLMs4OL and the instance / Text2Onto metrics,
4. a small headless agent run (`ontology-eval-agent.py` in-process),
5. tables and figures of the results present in `results/`.

In [ ]:
import env_setup  # backend/src + this folder on sys.path, agent .env
import json
import pandas as pd
from pathlib import Path

from builder.ontology import Ontology, OntoClass, OntoLink
from onto_eval import load_reference, schema_from_ontology, score_schema, text2onto_metrics, term_name
from datasets import DATASETS, llms4ol_gold_lists
from builder_eval import RunConfig, run_agent, score_session

## 1. References

In [ ]:
refs = {name: load_reference(name) for name in ("gutbrainie", "enron")}
pd.DataFrame({name: ref.summary() for name, ref in refs.items()})

In [ ]:
ref = refs["gutbrainie"]
pd.DataFrame([(l.label, l.source, l.target) for l in ref.links], columns=["link", "domain", "range"]).head(15)

A hand-made GutBrainIE ontology: synonyms (`Microbe`, `Disease`) should align semantically, the exact level only matches identical names.

In [ ]:
onto = Ontology(classes=[OntoClass(id="Bacteria"), OntoClass(id="Microbe"), OntoClass(id="Disease"),
                          OntoClass(id="Chemical"), OntoClass(id="Author")],
                links=[OntoLink(id="influences", source="Chemical", target="Disease"),
                       OntoLink(id="locatedIn", source="Bacteria", target="Disease"),
                       OntoLink(id="writes", source="Author", target="Chemical")])
score = score_schema(schema_from_ontology(onto), ref)
print(json.dumps(score.alignment, indent=1))
{k: round(v, 3) for k, v in score.metrics.items() if k.endswith("f1")}

In [ ]:
# checks: exact class matches are Bacteria/Chemical, "influence" (Chemical->DDF) is a valid triple
assert ("Bacteria", "Bacteria", 1.0) in score.alignment["class_exact"]
assert any(p.startswith("influences") for p, _, _ in score.alignment["rel_triple"])
assert not any(p.startswith("writes") for p, _, _ in score.alignment["rel_triple"])

## 2. User sessions (schema only)

In [ ]:
from builder_eval import score_saved_sessions, aggregate
users = score_saved_sessions()  # default: the backend's builder save_dir
users[["dataset", "run_id", "n_classes", "n_links", "class_sem_f1", "rel_triple_sem_f1", "schema_sem_f1"]]

In [ ]:
aggregate(users).round(3)

## 3. Gold annotations and instance metrics

In [ ]:
gb = DATASETS["gutbrainie"]
gold = gb.gold(ref)
doc = next(iter(gold.entities))
print(doc, sorted(gold.entities[doc])[:8], sorted(gold.relations[doc])[:4])

In [ ]:
# the gold scored against itself is perfect, with an identity alignment
from onto_eval import score_mentions
ident = {c.id: c.id for c in ref.classes}
m = score_mentions(gold, gold, ident, {l.id: l.id for l in ref.links})
assert all(abs(v - 1) < 1e-9 for k, v in m.items() if k.endswith("f1")), m
m

In [ ]:
# Text2Onto: the challenge's Jaccard >= 0.8 matching (greedy, one-to-one)
lists = llms4ol_gold_lists("scholarly")
print(len(lists["terms"]), len(lists["types"]), lists["types"][:5])
assert text2onto_metrics(lists["types"], lists["types"], "t")["t_f1"] == 1.0
text2onto_metrics(["hill range", "drumlin fields", "lake"], ["Hill Range", "drumlin field"], "t")

## 4. A small agent run

Needs the agent LLM (`.env`) and the GPU for the extraction model. LLMs4OL scholarly has 8 test documents, so the build is quick.

In [ ]:
cfg = RunConfig(DATASETS["llms4ol-scholarly"], "notebook-check", Path("results/notebook"), refine_turns=0,
                turn_build_timeout=120, final_build_timeout=300)
session, stats = run_agent(cfg)
print(stats["turns"], stats["build_complete"], stats["status"])
session.ontology.model_dump(exclude={"classes": {"__all__": {"x", "y"}}})

In [ ]:
metrics, details = score_session(cfg.dataset, session.ontology, session.graph, None, cfg.dataset.gold(None))
print({k: v for k, v in details["lists"].items()})
{k: round(v, 3) for k, v in metrics.items() if k.endswith(("f1", "n_pred"))}

## 5. Results, tables and figures

In [ ]:
import subprocess
print(subprocess.run(["./py.sh", "make-figures.py"], capture_output=True, text=True).stdout[-2000:])

In [ ]:
results = pd.concat([pd.read_csv(p).assign(file=p.name) for p in Path("results").glob("*.csv")], ignore_index=True)
results.groupby(["dataset", "source"])[[c for c in ["schema_sem_f1", "kg_ent_typed_len_f1", "a1_terms_f1", "a2_types_f1"] if c in results]].median()